# Evaluation of Operating-System Random Number Generation Mechanisms

This experiment captures **1 MiB (1,048,576 bytes) per available mechanism** and applies identical descriptive and statistical checks. Sources: Linux `/dev/urandom`, the OpenSSL `rand` command, and Windows CNG (`BCryptGenRandom` with `BCRYPT_USE_SYSTEM_PREFERRED_RNG`). Files are saved in `data/`; avoid committing or publishing raw random samples.

**Platform rule:** CNG is called only on Windows. This notebook never simulates it elsewhere and labels the result unavailable on Linux/macOS. `/dev/urandom` is labeled as a Linux result only on Linux. OpenSSL is run as an external executable. Run on Linux and Windows for the complete three-way comparison.

In [ ]:
from pathlib import Path
import sys, os, platform
HERE = Path.cwd() if (Path.cwd() / 'rng_evaluation.py').exists() else Path.cwd() / 'os_rng_evaluation'
HERE = HERE.resolve()
# This repository has a top-level `math/` package; avoid shadowing Python's standard `math` module.
sys.path[:] = [p for p in sys.path if os.path.realpath(p or os.getcwd()) != os.path.realpath(os.getcwd())]
sys.path.insert(0, str(HERE))
import numpy as np
import matplotlib.pyplot as plt
from rng_evaluation import SAMPLE_SIZE, generate_samples, analyze

OUT = HERE / 'data'
samples = generate_samples(OUT, SAMPLE_SIZE)
print(f'Host: {platform.platform()}')
for name, data in samples.items():
    state = f'{len(data):,} bytes captured' if data is not None else 'NOT RUN on this platform'
    print(f'{name}: {state}')
    if data is not None and len(data) != SAMPLE_SIZE:
        raise RuntimeError(f'{name} produced {len(data)} bytes; expected {SAMPLE_SIZE}')

In [ ]:
results = {name: analyze(data) for name, data in samples.items() if data is not None}
print(f'Analyzed {len(results)} available source(s).')

## Theoretical mechanism comparison

| Mechanism | Entropy inputs and initialization | Random-data generation | Practical notes |
|---|---|---|---|
| Linux kernel `/dev/urandom` | Kernel random subsystem gathers environmental noise (including device/event and interrupt timing) and mixes it into an internal pool. The kernel tracks initialization; modern kernels initialize the CRNG before normal output. | Kernel CRNG expands secret state (ChaCha20-based in modern Linux) to produce bytes. `getrandom()` is generally preferable where explicit initialization semantics matter. | Implementation varies by kernel version. This experiment identifies `/dev/urandom` only on Linux. |
| OpenSSL `rand` | OpenSSL's userspace DRBG is initialized from configured entropy sources, normally the OS CSPRNG; version, provider, and configuration matter. | The DRBG expands internal state and reseeds as required by its policy/provider. | A library layer over OS entropy, not an independent physical source. Record `openssl version -a`. |
| Windows CNG (`BCryptGenRandom`) | Windows maintains system cryptographic random state fed by platform entropy sources and managed by the OS. `BCRYPT_USE_SYSTEM_PREFERRED_RNG` selects the system-preferred provider without an application-managed algorithm handle. | CNG returns bytes from the OS cryptographic provider; internal construction and reseeding vary by Windows version. | This notebook calls native CNG only on Windows; it is not emulated elsewhere. |

All follow the CSPRNG pattern: gather/mix entropy, initialize secret state, then expand it efficiently while protecting and refreshing state. Exact entropy accounting and implementation details are platform/version specific; byte statistics cannot establish them.

## Byte-frequency distributions and histograms

The chi-square statistic compares observed counts for all 256 byte values with the uniform expectation (N/256). Its p-value assumes independent uniform bytes and is a diagnostic, not a security score.

In [ ]:
for name, result in results.items():
    counts = np.asarray(result['byte_counts'])
    fig, ax = plt.subplots(figsize=(11, 3.4))
    ax.bar(np.arange(256), counts, width=1.0, color='#3274a1')
    ax.axhline(result['bytes'] / 256, color='#c44e52', linestyle='--', label='Uniform expected count')
    ax.set(title=f'Byte frequency: {name}', xlabel='Byte value', ylabel='Count')
    ax.legend(); plt.tight_layout(); plt.show()
    print(f"Chi-square={result['byte_chi_square']:.2f}, df=255, p={result['byte_chi_square_p']:.4g}")

## Bit distribution, repeated patterns, and statistical tests

The monobit test checks the number of one bits against an equal-probability model. The runs test checks transitions between 0 and 1 and requires the bit proportion to be sufficiently close to 1/2. Autocorrelation uses centered byte values at lags 1–16; p-values use a normal approximation under independence. Repeated-pattern checks report the longest identical-byte run and duplicate aligned 4-byte blocks. Tests are approximate diagnostics; multiple lag checks increase false-positive chances.

In [ ]:
for name, r in results.items():
    print(f'\n{name}')
    print(f" bits: zeros={r['zeros']:,}, ones={r['ones']:,}, one fraction={r['bit_proportion_ones']:.6f}")
    print(f" monobit: z={r['monobit_z']:.3f}, p={r['monobit_p']:.4g}")
    print(f" runs: observed={r['bit_runs']:,}, expected={r['runs_expected']:.1f}, p={r['runs_p']:.4g}")
    print(f" repeated patterns: max same-byte run={r['max_repeated_byte_run']}, duplicate aligned 4-byte blocks={r['repeated_aligned_4byte_blocks']}")
    print(' autocorrelation (lag: r, p):')
    print({lag: (round(v['correlation'], 5), round(v['p_value'], 4)) for lag, v in r['autocorrelation'].items()})

## Comparison table

The table combines observed results with design/security characteristics. “No rejection” at 0.01 only means this sample did not trigger that test under its assumptions. Small p-values can occur by chance; investigate with independent fresh captures. Security characteristics describe the mechanism, not a property inferred from these statistics.

In [ ]:
rows = []
for name in ('Linux /dev/urandom', 'OpenSSL rand', 'Windows CNG'):
    if name not in results:
        rows.append({'Mechanism': name, 'Capture': 'Not run on this host', 'Byte chi-square p': '—', 'Monobit p': '—', 'Runs p': '—', 'Max byte run': '—', 'Security characteristics': 'Requires execution on its native host'})
        continue
    r = results[name]
    security = {'Linux /dev/urandom': 'Kernel CRNG; modern Linux uses ChaCha20-based generation; OS-managed state', 'OpenSSL rand': 'OpenSSL userspace DRBG, normally seeded from OS; provider/version dependent', 'Windows CNG': 'OS-managed system-preferred cryptographic RNG and entropy state'}[name]
    rows.append({'Mechanism': name, 'Capture': f"{r['bytes']:,} bytes", 'Byte chi-square p': f"{r['byte_chi_square_p']:.4g}", 'Monobit p': f"{r['monobit_p']:.4g}", 'Runs p': f"{r['runs_p']:.4g}", 'Max byte run': r['max_repeated_byte_run'], 'Security characteristics': security})
headers = list(rows[0])
print(' | '.join(headers)); print(' | '.join(['---'] * len(headers)))
for row in rows: print(' | '.join(str(row[h]) for h in headers))

## Why statistical success does not prove cryptographic security

A stream can have a nearly perfect histogram and pass monobit, runs, and autocorrelation tests while still being predictable. A deterministic generator with a public seed can produce statistically convincing output. Security depends on secret state, sufficient entropy at seeding, resistance to state recovery and prediction, safe reseeding, correct implementation, and protection against side channels or compromise. These tests inspect limited patterns in finite samples; they do not establish entropy, unpredictability, forward/backtracking resistance, or resistance to cryptanalysis. Use operating-system cryptographic APIs and rely on their documented properties and platform updates.

## Reproducibility

Each source is captured once per notebook execution; binary samples are not committed by default. Record OS/kernel, OpenSSL version/provider configuration, Python version, and date. Run on Linux and Windows to populate all three rows. macOS can test OpenSSL but cannot stand in for Linux `/dev/urandom` or Windows CNG.